# Part 1 (Mobile/Colab): EDA with Pandas — Youth Unemployment (World Bank)

**This notebook is optimized for Google Colab and mobile.**

**Prepared by [Olalekan Akinsande](https://www.linkedin.com/in/akinsande-olalekan/)**

### How to use on phone (Colab)
1. Open [colab.research.google.com](https://colab.research.google.com) in your browser (Android/iOS).
2. Tap **Upload** and select this notebook.
3. Run the first cell to **upload the CSV** from your phone.
4. Run cells in order.

**Dataset**: World Bank WDI — Youth unemployment (15–24), % of labor force.

## 0) Upload the CSV (from your phone)

In [ ]:
# Run this cell in Google Colab to upload the World Bank CSV from your phone.
try:
    from google.colab import files
    uploaded = files.upload()
    # Grab the first uploaded filename
    csv_path = list(uploaded.keys())[0]
    print('Uploaded:', csv_path)
except Exception as e:
    print('If not on Colab, set csv_path manually to your local file path.')
    csv_path = 'API_SL.UEM.1524.ZS_DS2_en_csv_v2_171560.csv'

## 1) Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
print('Libraries imported.')

## 2) Load the Data

In [ ]:
# World Bank CSVs usually have 4 metadata rows; we skip them.
df_raw = pd.read_csv(csv_path, skiprows=4)
print('Raw shape:', df_raw.shape)
df_raw.head(3)

## 3) Reshape Wide → Long

In [ ]:
to_drop = [c for c in df_raw.columns if c.startswith('Unnamed') or c in ['Indicator Name','Indicator Code']]
df = df_raw.drop(columns=to_drop)

df_long = df.melt(
    id_vars=['Country Name','Country Code'],
    var_name='Year',
    value_name='Youth_Unemployment'
)
df_long['Year'] = pd.to_numeric(df_long['Year'], errors='coerce')
df_long = df_long.dropna(subset=['Year'])
df_long['Year'] = df_long['Year'].astype(int)
df_long['Youth_Unemployment'] = pd.to_numeric(df_long['Youth_Unemployment'], errors='coerce')
print('Long shape after melt:', df_long.shape)
df_long.head()

## 4) Handle Missing Values (EDA-safe)

In [ ]:
df_long = df_long.dropna(subset=['Youth_Unemployment']).reset_index(drop=True)
print('Shape after dropping missing:', df_long.shape)

## 5) Summary Stats

In [ ]:
df_long['Youth_Unemployment'].describe()

## 6) Latest Year: Top 10 Highest

In [ ]:
latest_year = int(df_long['Year'].max())
latest = df_long[df_long['Year'] == latest_year]
top10 = latest.sort_values('Youth_Unemployment', ascending=False).head(10)
top10[['Country Name','Youth_Unemployment']].reset_index(drop=True)

## 7) World vs Sub-Saharan Africa Trend

In [ ]:
world_series = df_long[df_long['Country Name'] == 'World'].groupby('Year')['Youth_Unemployment'].mean()
ssa_series = df_long[df_long['Country Name'] == 'Sub-Saharan Africa'].groupby('Year')['Youth_Unemployment'].mean()

world_series.plot(label='World', linewidth=2)
ssa_series.plot(label='Sub-Saharan Africa', linewidth=2)
plt.title('Youth Unemployment: World vs Sub-Saharan Africa (15–24, % of labor force)')
plt.xlabel('Year')
plt.ylabel('%')
plt.legend()
plt.show()

## 8) Country Comparison: South Africa, Ethiopia, Nigeria, Kenya

In [ ]:
countries_to_plot = ['South Africa','Ethiopia','Nigeria','Kenya']
subset = df_long[df_long['Country Name'].isin(countries_to_plot)]

for c in countries_to_plot:
    s = subset[subset['Country Name'] == c].set_index('Year')['Youth_Unemployment'].sort_index()
    s.plot(label=c)

plt.title('Youth Unemployment Trend (Selected Countries)')
plt.xlabel('Year')
plt.ylabel('%')
plt.legend()
plt.show()

## 9) Save Clean Long Format (for Part 2 & 3)

In [ ]:
df_long.to_csv('youth_unemployment_long_clean.csv', index=False)
print('Saved → youth_unemployment_long_clean.csv')

**You have my best wishes!**